# Experiment 6: does a stranger history make the model less sure?

In Experiment 3 the model missed the cold-front drops, and its 10-90 % band usually did not reach down to what
happened. Here we make the **history** stranger, step by step, and watch how the width of the band (how unsure the
model is) changes. The coming drop itself is never shown to the model.

**Cases:** the Experiment 3 cases with a real drop of at least 2 °C (forecast starts 6 h before the drop,
10 weeks of history, temperature only).

**Two ways to make the history weird** (the last 24 h before the forecast are never changed, so the starting
level stays real):
1. **Inserted drops:** 1, 2, 4 or 8 artificial cold-front drops of 3 to 15 °C, spread evenly over the history.
   Their shape is the average cold front from Experiment 5 (fast fall, slow recovery).
2. **Noise:** random hour-to-hour noise with a strength (standard deviation) of 0.5 to 5 °C.

**Measure**
- **band width (°C):** mean width of the 10-90 % band over the 24 h (the price of a wider band: less precise)

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from tirex2 import TimeseriesType, load_model

DATA = Path("/workspace/data/austria_15km")
FIG = Path("/workspace/phase1-model-behavior/pair-1/experiments/figures/exp6")
FIG.mkdir(parents=True, exist_ok=True)
EXP3 = Path("/workspace/data/notebooks/results/exp3_forecasts_30perfront.npz")
EXP5 = Path("/workspace/data/notebooks/results/exp5_after_front.npz")
RESULTS = Path("/workspace/data/notebooks/results/exp6_weird_context.npz")
C, H, BEFORE = 10 * 168, 24, 6
DROP_N = [1, 2, 4, 8]
DROP_S = [3, 6, 9, 12, 15]
NOISE = [0.5, 1, 2, 3, 5]
KEEP_LAST = 24  # hours at the end of the history that are never changed
RERUN = False
QUICK = bool(os.environ.get("QUICK"))
if QUICK:  # quick code test
    FIG = Path("/tmp")
    EXP5 = Path("/tmp/exp5_after_front.npz") if Path("/tmp/exp5_after_front.npz").exists() else EXP5
    RESULTS = Path("/tmp/exp6_weird_context.npz")

In [2]:
df = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", "T2M"])
Tw = df.pivot(index="time", columns="point_id", values="T2M").interpolate(limit=3, limit_area="inside")
times, T = Tw.index, Tw.to_numpy(np.float32)
col = {p: i for i, p in enumerate(Tw.columns)}

z = np.load(EXP3)
real_drop = z["truth"][:, BEFORE] - z["truth"][:, BEFORE:BEFORE + 7].min(axis=1)
ok = real_drop >= 2
cases = pd.DataFrame({"point_id": z["point_id"][ok], "front": pd.to_datetime(z["front"][ok]),
                      "o": [times.get_loc(pd.Timestamp(s)) for s in z["start"][ok]]})
truth, base, real_drop = z["truth"][ok], z["A"][ok], real_drop[ok]
if QUICK:
    cases, truth, base, real_drop = cases.iloc[:6].reset_index(drop=True), truth[:6], base[:6], real_drop[:6]

# shape of the average cold front (Experiment 5): unusual temperature from the drop start to 48 h after, lowest point = -1
z5 = np.load(EXP5)
rel, med = z5["comp_rel"], z5["comp_T2M_median"]
shape = med[(rel >= 0) & (rel <= 48)] - med[(rel >= -6) & (rel < 0)].mean()
shape = (shape / -shape.min()).astype(np.float32)
print(f"{len(cases)} cases; inserted drop shape: lowest after {np.argmin(shape)} h, "
      f"{-shape[24]:.0%} of the drop left after 24 h, {-shape[-1]:.0%} after 48 h")

269 cases; inserted drop shape: lowest after 5 h, 80% of the drop left after 24 h, 62% after 48 h


In [3]:
def history(k):
    return T[cases.o[k] - C : cases.o[k], col[cases.point_id[k]]].copy()


def weird(h, kind, a, b, k):
    """kind 'drops': a drops of b °C; kind 'noise': noise of strength a °C."""
    h = h.copy()
    if kind == "drops":
        for s in np.linspace(168, C - KEEP_LAST - len(shape), a).astype(int):
            h[s : s + len(shape)] += b * shape
    elif kind == "noise":
        h[: C - KEEP_LAST] += np.random.default_rng(k).normal(0, a, C - KEEP_LAST).astype(np.float32)
    return h


SETUPS = {("none", 0, 0): "unchanged"}
SETUPS.update({("drops", n, s): f"{n} drop{'s' if n > 1 else ''} of {s} °C" for n in DROP_N for s in DROP_S})
SETUPS.update({("noise", a, 0): f"noise {a} °C" for a in NOISE})
key = lambda st: f"{st[0]}_{st[1]}_{st[2]}"

model = load_model("NX-AI/TiRex-2", device="cpu")
fc = {("none", 0, 0): base}
if RESULTS.exists() and not RERUN and not QUICK:
    zz = np.load(RESULTS)
    assert list(zz["point_id"]) == list(cases.point_id)
    fc.update({st: zz[key(st)] for st in SETUPS if key(st) in zz.files})
    print(f"loaded saved forecasts from {RESULTS.name}")
todo = [st for st in SETUPS if st not in fc]
t0 = time.time()
for st in todo:
    ts = [TimeseriesType(torch.from_numpy(weird(history(k), *st, k)[None]), None, None) for k in range(len(cases))]
    out = model.forecast(ts, prediction_length=H, output_type="numpy", batch_size=64)
    fc[st] = np.stack([q[0] for q in out])
if todo:
    print(f"{len(todo)} setups x {len(cases)} cases in {time.time() - t0:.0f} s")
    np.savez_compressed(RESULTS, point_id=cases.point_id.to_numpy(), front=cases.front.astype(str).to_numpy().astype("U"),
                        start=np.asarray(times[cases.o].astype(str)).astype("U"), truth=truth, shape=shape,
                        **{key(st): v for st, v in fc.items() if st[0] != "none"})

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

loaded saved forecasts from exp6_weird_context.npz


## Results

In [4]:
def width(st):
    return (fc[st][:, 8] - fc[st][:, 0]).mean(axis=1)


res = pd.DataFrame([{"setup": SETUPS[st], "kind": st[0], "a": st[1], "b": st[2], "width": width(st).mean()} for st in SETUPS])
print(f"unchanged history: band width {res.width[0]:.1f} °C")

unchanged history: band width 6.1 °C


### Inserted drops: does more weirdness always widen the band?

x = size of each inserted drop (0 = unchanged history), one line per number of
inserted drops. A line that rises and then falls again = the band is widest for medium drops.

In [5]:
LINE_COL = {1: "#deebf7", 2: "#9ecae1", 4: "#6baed6", 8: "#3182bd"}
xs0 = [0] + DROP_S
fig = go.Figure()
for n in DROP_N:
    r = res.query("kind == 'drops' and a == @n").sort_values("b")
    fig.add_trace(go.Scatter(x=xs0, y=np.r_[res.width[0], r.width], mode="lines+markers", name=f"{n} inserted drop{'s' if n > 1 else ''}",
                             line={"color": LINE_COL[n], "width": 2.5}, marker={"size": 7, "color": LINE_COL[n]},
                             hovertemplate=f"{n} drops of %{{x}} °C: band %{{y:.1f}} °C<extra></extra>"))
fig.update_xaxes(title_text="size of each inserted drop (°C, 0 = unchanged)", tickvals=xs0)
fig.update_yaxes(title_text="band width (°C)")
fig.update_layout(height=430, template="plotly_dark", legend={"orientation": "h", "y": -0.25},
                  title=f"Does a stranger history make the model less sure?<br><sup>{len(cases)} cold-front cases; one line per number of "
                        "inserted drops; the band is widest for medium drops and narrows again for big or many drops</sup>")
fig.write_html(FIG / "exp6_drops_lines.html", include_plotlyjs="cdn")
fig.show()

### Play with the history: one example case

The case with the median real drop. Move the slider to change the size of 4 inserted drops (0 = unchanged history);
top = the 10-week history the model gets, bottom = the last 3 days, the forecast band and what happened (white).
Every slider position is a real forecast (steps of 0.25 °C, no interpolation between steps); the lines glide between
neighbouring steps only as an animation.

In [6]:
SLIDE_N, SLIDE_S = 4, np.round(np.arange(0, 15.001, 0.25), 2)
SLIDE_RES = RESULTS.with_name("exp6_slider_fine.npz")
ex = int(np.argsort(real_drop)[len(real_drop) // 2])
o, c = cases.o[ex], col[cases.point_id[ex]]
ht, ft = times[o - C : o], times[o : o + H]
hist_s = np.stack([weird(history(ex), "drops", SLIDE_N, s, ex) for s in SLIDE_S])
if SLIDE_RES.exists() and not RERUN and not QUICK:
    zs = np.load(SLIDE_RES)
    assert zs["point_id"] == cases.point_id[ex] and np.allclose(zs["sizes"], SLIDE_S)
    fc_s = zs["forecast"]
    print(f"loaded saved slider forecasts from {SLIDE_RES.name}")
else:
    t0 = time.time()
    out = model.forecast([TimeseriesType(torch.from_numpy(h[None]), None, None) for h in hist_s], prediction_length=H,
                         output_type="numpy", batch_size=64)
    fc_s = np.stack([q[0] for q in out])
    print(f"{len(SLIDE_S)} slider forecasts in {time.time() - t0:.0f} s")
    if not QUICK:
        np.savez_compressed(SLIDE_RES, point_id=cases.point_id[ex], start=str(times[o]), sizes=SLIDE_S, n_drops=SLIDE_N,
                            history=hist_s.astype(np.float32), forecast=fc_s, truth=truth[ex])
print(f"point {cases.point_id[ex]}, front {cases.front[ex]:%Y-%m-%d}, forecast from {times[o]:%Y-%m-%d %H:%M} UTC, "
      f"real drop {real_drop[ex]:.1f} °C; size 0 vs saved unchanged forecast: max diff {np.abs(fc_s[0] - base[ex]).max():.3f} °C")

loaded saved slider forecasts from exp6_slider_fine.npz
point 1785, front 2026-08-28, forecast from 2026-08-28 17:00 UTC, real drop 6.0 °C; size 0 vs saved unchanged forecast: max diff 0.000 °C


In [7]:
def traces(i):
    h, f = hist_s[i], fc_s[i]
    return [go.Scatter(x=ht, y=h, line={"color": "#9a9a9a", "width": 1}, name="history", showlegend=False),
            go.Scatter(x=ht[-72:], y=h[-72:], line={"color": "#9a9a9a", "width": 1.5}, name="history", showlegend=False),
            go.Scatter(x=ft, y=f[0], mode="lines", line={"width": 0, "color": "#6baed6"}, showlegend=False, hoverinfo="skip"),
            go.Scatter(x=ft, y=f[8], mode="lines", line={"width": 0, "color": "#6baed6"}, fill="tonexty",
                       fillcolor="rgba(107,174,214,0.3)", name="forecast band (10-90 %)", hoverinfo="skip"),
            go.Scatter(x=ft, y=f[4], mode="lines", line={"color": "#6baed6", "width": 2.5}, name="forecast (median)")]


def title(i):
    lab = "unchanged history" if SLIDE_S[i] == 0 else f"4 inserted drops of {SLIDE_S[i]:g} °C"
    return f"History with inserted drops: <b>{lab}</b>"


fig = make_subplots(rows=2, cols=1, vertical_spacing=0.12, row_heights=[0.4, 0.6],
                    subplot_titles=["History the model gets (10 weeks)", "Last 3 days, forecast and what happened"])
for t, r in zip(traces(0), [1, 2, 2, 2, 2]):
    fig.add_trace(t, row=r, col=1)
fig.add_trace(go.Scatter(x=ft, y=truth[ex], mode="lines", line={"color": "white", "width": 3}, name="what happened"), row=2, col=1)
fig.frames = [go.Frame(name=f"{s:g}", data=traces(i), traces=[0, 1, 2, 3, 4], layout={"title_text": title(i)})
              for i, s in enumerate(SLIDE_S)]
anim = {"mode": "immediate", "frame": {"duration": 0, "redraw": False}, "transition": {"duration": 60, "easing": "linear"}}
steps = [{"method": "animate", "label": f"{s:g}" if s % 3 == 0 else "", "args": [[f"{s:g}"], anim]} for s in SLIDE_S]
pad = 1.0  # fixed axes, so only the lines move
lo2 = min(hist_s[:, -72:].min(), fc_s[:, 0].min(), truth[ex].min()) - pad
hi2 = max(hist_s[:, -72:].max(), fc_s[:, 8].max(), truth[ex].max()) + pad
fig.update_yaxes(title_text="T2M (°C)")
fig.update_yaxes(range=[hist_s.min() - pad, hist_s.max() + pad], row=1, col=1)
fig.update_yaxes(range=[lo2, hi2], row=2, col=1)
fig.update_layout(height=700, template="plotly_dark", legend={"orientation": "h", "y": -0.25}, title=title(0),
                  sliders=[{"active": 0, "steps": steps, "pad": {"t": 40}, "currentvalue": {"prefix": "size of each inserted drop: ",
                                "suffix": " °C", "visible": True}}])
fig.write_html(FIG / "exp6_slider_drops.html", include_plotlyjs="cdn", auto_play=False)
fig.show()

## Reusing the results

Saved in `data/notebooks/results/exp6_weird_context.npz` (the unchanged-history forecasts are setup A of
Experiment 3):

```python
z = np.load("/workspace/data/notebooks/results/exp6_weird_context.npz")
z["point_id"], z["front"], z["start"], z["truth"]  # one entry per case; the drop starts at hour index 6
z["shape"]                                         # inserted drop shape (49 h, lowest point = -1)
z["drops_4_9"]   # (cases, 9 quantiles, 24 h): 4 inserted drops of 9 °C; drops_<n>_<size> for all combinations
z["noise_2_0"]   # noise of 2 °C; noise_<strength>_0

zs = np.load("/workspace/data/notebooks/results/exp6_slider_fine.npz")   # slider: one case, 4 inserted drops
zs["sizes"], zs["history"], zs["forecast"], zs["truth"]  # 61 sizes 0-15 °C; (61, 1680), (61, 9, 24), (24,)
```